# UFC Data Exploration

**Project:** End-to-end data pipeline on UFC fights  
**Notebook:** 01 — Initial data exploration  
**Author:** Kris LOKOUN  
**Date:** June 2026

## Goal
Explore the raw UFC fight-results dataset to understand its structure,
volume, and quality before building the pipeline.

## Data source
Public data scraped from ufcstats.com, refreshed daily.
Repository: `Greco1899/scrape_ufc_stats` (GPL-d, no API key. ni clé d'API.

## 1. Import the library and connect to the data

We use DuckDB , an in-process analytical database that can read
a remote CSV file directly from its URL.

In [11]:
import duckdb

duckdb.sql("""
    SELECT COUNT(*) AS n_fights
    FROM 'https://raw.githubusercontent.com/Greco1899/scrape_ufc_stats/main/ufc_fight_results.csv'
""")

┌──────────┐
│ n_fights │
│  int64   │
├──────────┤
│     8701 │
└──────────┘

The dataset contains **8701 fights**, covering the full history of the UFC more than enough data for analysis and modeling.

## 2. Preview the data

We display the first rows to discover the available columns and the type
of information we have for each fight.

In [20]:
duckdb.sql("""
    SELECT *
    FROM 'https://raw.githubusercontent.com/Greco1899/scrape_ufc_stats/main/ufc_fight_results.csv'
    LIMIT 5
""")

┌───────────────────────────────────┬──────────────────────────────────────────┬─────────┬────────────────────┬───────────────────────┬───────┬──────────┬───────────────────┬───────────────┬───────────────────────────────────────────────────────────────────────┬────────────────────────────────────────────────────┐
│               EVENT               │                   BOUT                   │ OUTCOME │    WEIGHTCLASS     │        METHOD         │ ROUND │   TIME   │    TIME FORMAT    │    REFEREE    │                                DETAILS                                │                        URL                         │
│              varchar              │                 varchar                  │ varchar │      varchar       │        varchar        │ int64 │   time   │      varchar      │    varchar    │                                varchar                                │                      varchar                       │
├───────────────────────────────────┼───────────────

### Observations on the raw data

The dataset has 11 columns. Key fields for the project:

- **OUTCOME** is encoded as `W/L` / `L/W`, indicating whether the first or
  second fighter won. This will need to be transformed into a clear winner.
- **BOUT** holds both fighters in a single field ("A vs. B"); it will have
  to be split into two separate columns.
- **METHOD**, **ROUND**, and **WEIGHTCLASS** are promising features for
  later analysis and modeling.

## 3. Inspect the schema

We list every column and its data type to get a clean overview of the
table structure.


In [22]:
duckdb.sql("""
    DESCRIB
    SELECT *
    FROM 'https://raw.githubusercontent.com/Greco1899/scrape_ufc_stats/main/ufc_fight_results.csv'
""")

┌─────────────┬─────────────┬─────────┬─────────┬─────────┬─────────┐
│ column_name │ column_type │  null   │   key   │ default │  extra  │
│   varchar   │   varchar   │ varchar │ varchar │ varchar │ varchar │
├─────────────┼─────────────┼─────────┼─────────┼─────────┼─────────┤
│ EVENT       │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ BOUT        │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ OUTCOME     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ WEIGHTCLASS │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ METHOD      │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ ROUND       │ BIGINT      │ YES     │ NULL    │ NULL    │ NULL    │
│ TIME        │ TIME        │ YES     │ NULL    │ NULL    │ NULL    │
│ TIME FORMAT │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ REFEREE     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ DETAILS     │ VARCHAR     │ YES     │ NULL    │ NULL    │ NULL    │
│ URL         │ VARC

### Schema overview

The table has 11 columns: mostly `VARCHAR` (text), with `ROUND` as an
integer (`BIGINT`) and `TIME` as a time value. Every column allows NULLs,
so we need to check how many missing values actually exist before modeling.

## 4. Check for missing values

We count, for each important column, how many rows are empty (NULL).
This tells us about the quality and completeness of the dataset.## 4. Check for missing values

We count, for each important column, how many rows are empty (NULL).
This tells us about the quality and completeness of the dataset.

In [23]:
duckdb.sql("""
    SELECT COUNT(*) AS total_rows,
           COUNT(*) - COUNT(OUTCOME)  AS missing_outcome,
           COUNT(*) - COUNT(METHOD)   AS missing_method,
           COUNT(*) - COUNT(ROUND)    AS missing_round,
           COUNT(*) - COUNT(REFEREE)  AS missing_referee
    FROM 'https://raw.githubusercontent.com/Greco1899/scrape_ufc_stats/main/ufc_fight_results.csv'
""")

┌────────────┬─────────────────┬────────────────┬───────────────┬─────────────────┐
│ total_rows │ missing_outcome │ missing_method │ missing_round │ missing_referee │
│   int64    │      int64      │     int64      │     int64     │      int64      │
├────────────┼─────────────────┼────────────────┼───────────────┼─────────────────┤
│       8701 │               0 │              0 │             0 │              26 │
└────────────┴─────────────────┴────────────────┴───────────────┴─────────────────┘

### Data quality conclusion

Out of 8,701 fights, the critical columns (OUTCOME, METHOD, ROUND) have
**zero missing values** : the dataset is complete where it matters most.
Only REFEREE has 26 missing values (0.3%), which is negligible and
irrelevant for prediction. Overall, the data quality is excellent and
ready for transformation.

## 5. First transformation: extract the winner

`BOUT` contains both fighters ("Fighter A vs. Fighter B") and `OUTCOME`
tells us which side won (`W/L` = first fighter, `L/W` = second fighter).
We combine them to build a clear `winner` column.

In [24]:
duckdb.sql("""
    SELECT
        BOUT,
        OUTCOME,
        split_part(BOUT, ' vs. ', 1) AS fighter_1,
        split_part(BOUT, ' vs. ', 2) AS fighter_2,
        CASE
            WHEN OUTCOME = 'W/L' THEN split_part(BOUT, ' vs. ', 1)
            WHEN OUTCOME = 'L/W' THEN split_part(BOUT, ' vs. ', 2)
            ELSE NULL
        END AS winner
    FROM 'https://raw.githubusercontent.com/Greco1899/scrape_ufc_stats/main/ufc_fight_results.csv'
    LIMIT 10
""")

┌──────────────────────────────────────────┬─────────┬──────────────────────┬───────────────────────┬────────────────────┐
│                   BOUT                   │ OUTCOME │      fighter_1       │       fighter_2       │       winner       │
│                 varchar                  │ varchar │       varchar        │        varchar        │      varchar       │
├──────────────────────────────────────────┼─────────┼──────────────────────┼───────────────────────┼────────────────────┤
│ Arnold Allen vs. Melquizael Costa        │ W/L     │ Arnold Allen         │ Melquizael Costa      │ Arnold Allen       │
│ Dooho Choi vs. Daniel Santos             │ W/L     │ Dooho Choi           │ Daniel Santos         │ Dooho Choi         │
│ Malcolm Wellmaker vs. Juan Diaz          │ L/W     │ Malcolm Wellmaker    │ Juan Diaz             │ Juan Diaz          │
│ Modestas Bukauskas vs. Christian Edwards │ W/L     │ Modestas Bukauskas   │ Christian Edwards     │ Modestas Bukauskas │
│ Timmy Cuamba v

## 6. Load the raw data into a DuckDB database

We move from ad-hoc exploration to the first pipeline step: storing the raw
fight results in a persistent DuckDB file on disk (the "raw" layer).
Following the ELT approach, we load the data untransformed and will clean
it later in the transformation layer.

In [27]:
import duckdb

# Connect to our project database (created on disk if it doesn't exist)
con = duckdb.connect("ufc.duckdb")

# Base URL of the public UFC data repository
base_url = "https://raw.githubusercontent.com/Greco1899/scrape_ufc_stats/main/"

# The raw files to load: table name -> CSV file name
sources = {
    "raw_fight_results": "ufc_fight_results.csv",
    "raw_fight_stats":   "ufc_fight_stats.csv",
    "raw_event_details": "ufc_event_details.csv",
    "raw_fighter_tott":  "ufc_fighter_tott.csv",
}

# Load each CSV into its own raw table, reading every column as text
for table_name, file_name in sources.items():
    con.sql(f"""
        CREATE OR REPLACE TABLE {table_name} AS
        SELECT * FROM read_csv('{base_url}{file_name}', all_varchar = true)
    """)
    print(f"Loaded {table_name}")

print("All raw tables created in ufc.duckdb")

Loaded raw_fight_results
Loaded raw_fight_stats
Loaded raw_event_details
Loaded raw_fighter_tott
All raw tables created in ufc.duckdb


In [28]:
con.sql("SHOW TABLES")

┌───────────────────┐
│       name        │
│      varchar      │
├───────────────────┤
│ raw_event_details │
│ raw_fight_results │
│ raw_fight_stats   │
│ raw_fighter_tott  │
└───────────────────┘

## 7. Sanity check: row count per raw table

We count the rows in each raw table to confirm every CSV was fully loaded.
This is a basic data-quality check before moving to the transformation layer.

In [29]:
for table_name in ["raw_event_details", "raw_fight_results",
                   "raw_fight_stats", "raw_fighter_tott"]:
    n = con.sql(f"SELECT COUNT(*) AS n FROM {table_name}").fetchone()[0]
    print(table_name, "->", n, "rows")

raw_event_details -> 774 rows
raw_fight_results -> 8701 rows
raw_fight_stats -> 40998 rows
raw_fighter_tott -> 4496 rows


### Raw layer validation

All four raw tables loaded successfully with consistent volumes:

- `raw_event_details`: 774 events
- `raw_fight_results`: 8,701 fights (~11 fights per event)
- `raw_fight_stats`: 40,998 rows (per fighter, per round — finest grain)
- `raw_fighter_tott`: 4,496 fighters

The row counts are coherent with the UFC's structure, confirming the data
was fully and correctly ingested. The raw layer is complete and ready for
the transformation step.